# Leksjon 1: Python Tenker Som en Hacker

**Velkommen!** I denne leksjonen skal vi lære det grunnleggende i Python-programmering ved å bruke det som et cybersikkerhetsverktøy. I stedet for bare å skrive "Hello World", skal vi lære hvordan hackere og sikkerhetseksperter ser på nettet.

## 1. Introduksjon: Angriperens vs. Forsvarerens Tankesett
Cybersikkerhet handler ikke bare om verktøy; det handler om et tankesett. 
- **Angripere** leter etter **ett** svakt punkt for å komme seg inn.
- **Forsvarere** må sikre **alle** punkter.

**Etikk:** Ferdighetene du lærer her er kraftige. Uautorisert testing av systemer du ikke eier er ulovlig og uetisk. Ha alltid tillatelse før du skanner eller tester et nettsted. Vi bruker trygge, pedagogiske mål i dag.

## 2. Python Grunnleggende: Variabler og Tekst
I Python lagrer vi data i **variabler**. Tekst kalles en **String** (streng).

La oss se på et enkelt eksempel der vi lagrer en URL.

In [ ]:
# Lage en variabel
target_url = "https://example.com"

# Skrive ut variabelen
print("Mål:", target_url)

### Lister og Løkker
Ofte vil vi jobbe med flere ting samtidig (som en liste med passord eller URL-er). Vi bruker **Lister** for lagring og **Løkker** (Loops) for å gå gjennom dem.

In [ ]:
# En liste over vanlige admin-stier
admin_paths = ["/admin", "/login", "/dashboard", "/config"]

# Gå gjennom listen med en løkke
for path in admin_paths:
    full_url = target_url + path
    print("Sjekker potensiell sti:", full_url)

## 3. `requests`-biblioteket
Nettlesere som Chrome eller Firefox sender **HTTP-forespørsler** til servere for å hente nettsider. Python kan også gjøre dette ved hjelp av `requests`-biblioteket. Slik samhandler skript med nettet.

In [ ]:
import requests
import urllib3

# Undertrykk advarselen om at vi gjør en ubekreftet HTTPS-forespørsel
# I produksjon ville du fikset sertifikatproblemet i stedet for å undertrykke det.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Send en GET-forespørsel til målet
# verify=False brukes ofte i sikkerhetstesting for å omgå SSL-feil fra proxyer eller selvsignerte sertifikater
response = requests.get(target_url, verify=False)

# Sjekk statuskoden (200 betyr OK)
print("Statuskode:", response.status_code)

# Begrens utskriften til de første 500 tegnene så vi ikke spammer skjermen
print("Forhåndsvisning av sideinnhold:\n", response.text[:500])

## 4. Lese HTTP-overskrifter (Headers)
Når en server svarer, sender den **Headers** (metadata) før selve HTML-innholdet. Disse overskriftene kan fortelle oss mye om sikkerheten til serveren.

Vanlige sikkerhetsoverskrifter vi ser etter:
- `X-Frame-Options`: Hindrer Clickjacking.
- `Strict-Transport-Security` (HSTS): Tvinger HTTPS.
- `Content-Security-Policy` (CSP): Hindrer XSS.

La oss inspisere overskriftene til målet vårt.

In [ ]:
# Få tilgang til overskrifter (headers) fra responsobjektet
headers = response.headers

# Skriv ut alle overskrifter pent
for key, value in headers.items():
    print(f"{key}: {value}")

print("-" * 30)

# Sjekk etter en spesifikk sikkerhetsoverskrift
security_header = "X-Frame-Options"

if security_header in headers:
    print(f"[+] Fant {security_header}: {headers[security_header]}")
else:
    print(f"[-] Mangler {security_header} - Potensiell sårbarhet!")

## 5. Introduksjon til XSS (Cross-Site Scripting)

**XSS** oppstår når en angriper kan injisere ondsinnet kode (vanligvis JavaScript) inn i et nettsted som andre brukere ser.

Det mest grunnleggende tegnet på XSS-sårbarhet er når en side reflekterer brukerinndata uten å rense det ("sanitize"). Vi ser ofte etter `<script>`-taggen som et bevis på konseptet (Proof of Concept).

La oss prøve å finne en `<script>`-tag i en bit med HTML-kode.

In [ ]:
# Simulert HTML-innhold (tenk deg at dette kom fra et nettsted)
html_content = """
<html>
  <body>
    <h1>Velkommen!</h1>
    <p>Brukerkommentar: <script>alert('Hacked!');</script></p>
  </body>
</html>
"""

# Manuelt søk med Python
if "<script>" in html_content:
    print("FARE: Mulig XSS-vektor funnet! <script>-tag oppdaget.")
else:
    print("Trygt: Ingen <script>-tagger funnet.")

### Din Tur: Automatiser Skanneren

Kombiner det vi har lært! Skriv et skript nedenfor som:
1. Henter en URL (spør brukeren om inndata eller bruk en variabel).
2. Sjekker om `Content-Security-Policy` mangler.
3. Sjekker om teksten "\<script>" vises i innholdet (forenklet sjekk).

*(Merk: Virkelige skannere er mye mer komplekse, men dette er logikken!)*

In [ ]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Mål-URL (endre til et trygt/tillatt mål)
target_url = "https://example.com"

try:
    response = requests.get(target_url, verify=False)

    # 2. Sjekk om Content-Security-Policy-overskriften mangler
    if "Content-Security-Policy" not in response.headers:
        print("[-] MANGLER overskrift: Content-Security-Policy — XSS-beskyttelse kan være svak!")
    else:
        print(f"[+] Fant Content-Security-Policy: {response.headers['Content-Security-Policy']}")

    # 3. Sjekk om <script>-tag vises i sideinnholdet
    if "<script>" in response.text.lower():
        print("[!] ADVARSEL: <script>-tag funnet i sideinnholdet — krever manuell verifisering for XSS!")
    else:
        print("[+] Ingen inline <script>-tagger oppdaget i innholdet.")

except Exception as e:
    print(f"[FEIL] Kunne ikke hente URL: {e}")

## Oppsummering

I dag lærte vi:
1. **Variabler og Løkker** lar oss automatisere repetitive oppgaver.
2. **Requests** lar Python snakke med nettet.
3. **Headers** avslører sikkerhetskonfigurasjoner.
4. Små tekstmønstre (som `<script>`) kan indikere store sårbarheter.

**Neste Leksjon:** Vi skal dykke ned i filer, logger og digital etterforskning (forensics)!

In [ ]:
# Løsning for "Din Tur"-oppgaven
# Fjern kommentaren (#) for å se løsningen

# url = input("Skriv inn URL å skanne: ")
# try:
#     r = requests.get(url)
#     if "Content-Security-Policy" not in r.headers:
#         print("[-] Mangler Content-Security-Policy header")
#     if "<script>" in r.text:
#         print("[!] <script>-tag funnet i kroppen (krever manuell verifisering)")
# except Exception as e:
#     print("Feil ved henting av URL:", e)

---

##  Refleksjon — Hva jeg lærte i dag

I denne leksjonen ble jeg introdusert for grunnleggende Python-programmering gjennom et cybersikkerhetsperspektiv. I stedet for å lære syntaks isolert, ble hvert konsept brukt direkte i en reell sikkerhetsoppgave, noe som gjorde materialet både engasjerende og meningsfullt.

### Viktige lærdommer

**Python-grunnleggende**
Jeg lærte å lagre data i variabler og jobbe med tekst (strings) og lister. Løkkestrukturen lot meg automatisere repetitive oppgaver — for eksempel å iterere over en liste med stier for å bygge opp flere URL-er automatisk. Dette demonstrerte umiddelbart hvorfor automatisering er sentralt i sikkerhetsarbeid.

**HTTP og `requests`-biblioteket**
Jeg oppdaget at Python kan kommunisere med nettet på samme måte som en nettleser, ved å sende HTTP GET-forespørsler og motta svar. Å forstå strukturen til et HTTP-svar — statuskode, overskrifter og innhold — ga meg en klarere forståelse av hvordan nettkommunikasjon fungerer på teknisk nivå.

**Sikkerhetsoverskrifter (Security Headers)**
Jeg lærte at HTTP-svarøverskrifter inneholder viktig sikkerhetsmetadata. Fraværet av overskrifter som `Content-Security-Policy` eller `X-Frame-Options` kan eksponere en nettapplikasjon for angrep som XSS eller Clickjacking. Å sjekke disse overskriftene programmatisk er et standardtrinn i sikkerhetsvurderinger.

**Cross-Site Scripting (XSS)**
Jeg ble introdusert for XSS som en sårbarhetskategori der urensede brukerinput lar ondsinnede skript kjøre i et offers nettleser. Ved å søke etter `<script>`-tagger i HTML-innhold utførte jeg en forenklet, men konseptuelt korrekt automatisert deteksjon.

**Bygge skanneren**
Ved å kombinere alle konseptene ovenfor skrev jeg en automatisert skanner som henter en URL, sjekker om en sikkerhetsoverskrift mangler, og markerer mulige XSS-vektorer. Denne øvelsen viste hvordan noen få linjer Python kan gjenskape kjernlogikken i profesjonelle sikkerhetsverktøy.

### Personlig refleksjon
Det som overrasket meg mest var hvor raskt Python lar deg gå fra et konsept til et fungerende verktøy. Angriperens og forsvarerens tankesett som ble introdusert i starten av leksjonen endret også hvordan jeg tenker på kode: alle inndataer er potensielt upålitelige, og enhver manglende konfigurasjon er en potensiell svakhet. Jeg føler meg trygg på å anvende disse konseptene i neste leksjon.